In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('dataset.csv')
df.head()

,description,category,silhouette,fabric,neckline,sleeve,length,embellishment,color
0,Floor length chiffon bridesmaid dress with ple...,Bridesmaid Dress,Not Specified,Chiffon,V-neck,Not Specified,Floor Length,Pleated,Sage|Dusty Blue
1,Sparkly sequin fitted prom gown featuring a de...,Prom Gown,Sheath,Sequin,Illusion,Not Specified,Not Specified,Sequin,Not Specified
2,Off shoulder satin ball gown with corset bodic...,Wedding Dress,Ball Gown,Satin,Off-shoulder,One-shoulder Drape,Sweep Train,Corset Bodice,Royal Navy
3,Lace mermaid wedding dress with long sleeves a...,Wedding Dress,Mermaid,Lace,Not Specified,Long Sleeve,Floor Length,Lace Overlay|Scalloped Hem,Ivory
4,Short cocktail dress with feather trim and bea...,Cocktail Dress,Not Specified,Not Specified,Not Specified,Not Specified,Short/Mini,Feather Trim|Beaded,Not Specified


In [ ]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)
df[["description", "clean_description"]].head()

,description,clean_description
0,Floor length chiffon bridesmaid dress with ple...,floor length chiffon bridesmaid dress with ple...
1,Sparkly sequin fitted prom gown featuring a de...,sparkly sequin fitted prom gown featuring a de...
2,Off shoulder satin ball gown with corset bodic...,off shoulder satin ball gown with corset bodic...
3,Lace mermaid wedding dress with long sleeves a...,lace mermaid wedding dress with long sleeves a...
4,Short cocktail dress with feather trim and bea...,short cocktail dress with feather trim and bea...


In [ ]:
df["embellishment"] = df["embellishment"].apply(lambda x: x.split("|"))
df["color"] = df["color"].apply(lambda x: x.split("|"))

In [ ]:
print(df["clean_description"].sample(5).tolist())
print(df["color"].sample(5).tolist())


['fitted jersey cocktail dress with v neckline and side slit in black', 'fitted sequin mermaid gown with halter neckline and thigh high slit in black', 'lace overlay cocktail dress with scoop neckline and short sleeves in emerald', 'sequin embellished column gown with one shoulder neckline in silver', 'crepe sheath dress with square neckline and cap sleeves in emerald']
[['Dusty Blue'], ['Dusty Blue'], ['Silver'], ['Gold'], ['Gold']]


In [ ]:
print(df["embellishment"].sample(5).tolist())

[['Feather Trim', 'Beaded'], ['Feather Trim'], ['Sequin'], ['Lace Overlay', 'Scalloped Hem'], ['Pleated']]


In [ ]:
df.to_csv("dataset_clean.csv", index=False)

In [ ]:
print(type(df["color"].iloc[0]))

<class 'list'>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_description"])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
single_label_attrs = ["category", "silhouette", "fabric", "neckline", "sleeve", "length"]
single_label_models = {}
for attr in single_label_attrs:
    y = df[attr]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    single_label_models[attr] = model
    print(attr, "trained. Train accuracy:", model.score(X_train, y_train))

category trained. Train accuracy: 0.9318181818181818
silhouette trained. Train accuracy: 0.8181818181818182
fabric trained. Train accuracy: 0.8863636363636364
neckline trained. Train accuracy: 0.9090909090909091
sleeve trained. Train accuracy: 0.8636363636363636
length trained. Train accuracy: 0.6818181818181818


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
multi_label_attrs = ["embellishment", "color"]
multi_label_models = {}
multi_label_binarizers = {}
for attr in multi_label_attrs:
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df[attr])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    model.fit(X_train, y_train)
    multi_label_models[attr] = model
    multi_label_binarizers[attr] = mlb
    print(attr, "trained.")

embellishment trained.
color trained.


In [ ]:
import joblib
import os
os.makedirs("models", exist_ok=True)
joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(single_label_models, "models/single_label_models.pkl")
joblib.dump(multi_label_models, "models/multi_label_models.pkl")
joblib.dump(multi_label_binarizers, "models/multi_label_binarizers.pkl")
print("All models saved to /models")

All models saved to /models


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
single_label_results = []
for attr in single_label_attrs:
    y = df[attr]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = single_label_models[attr]
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    single_label_results.append({"attribute": attr, "accuracy": acc, "f1_score": f1})
single_label_df = pd.DataFrame(single_label_results)
print(single_label_df)

    attribute  accuracy  f1_score
0    category  0.727273  0.628099
1  silhouette  0.363636  0.223776
2      fabric  0.454545  0.353535
3    neckline  0.727273  0.721212
4      sleeve  0.454545  0.450000
5      length  0.363636  0.216783


In [ ]:
multi_label_results = []
for attr in multi_label_attrs:
    mlb = multi_label_binarizers[attr]
    y = mlb.fit_transform(df[attr])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = multi_label_models[attr]
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)  # exact match accuracy across all labels
    f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
    multi_label_results.append({"attribute": attr, "accuracy": acc, "f1_score": f1})
multi_label_df = pd.DataFrame(multi_label_results)
print(multi_label_df)

       attribute  accuracy  f1_score
0  embellishment       0.0       0.0
1          color       0.0       0.0


In [ ]:
all_results = pd.concat([single_label_df, multi_label_df], ignore_index=True)
print(all_results)
overall_f1 = all_results["f1_score"].mean()
print("Overall average F1 score across all attributes:", overall_f1)

       attribute  accuracy  f1_score
0       category  0.727273  0.628099
1     silhouette  0.363636  0.223776
2         fabric  0.454545  0.353535
3       neckline  0.727273  0.721212
4         sleeve  0.454545  0.450000
5         length  0.363636  0.216783
6  embellishment  0.000000  0.000000
7          color  0.000000  0.000000
Overall average F1 score across all attributes: 0.3241757611075793


In [ ]:
# Check if the model is predicting empty label sets
for attr in multi_label_attrs:
    model = multi_label_models[attr]
    mlb = multi_label_binarizers[attr]
    X_train, X_test, y_train, y_test = train_test_split(X, mlb.transform(df[attr]), test_size=0.2, random_state=42)
    y_pred = model.predict(X_test)
    print(attr, "- avg labels predicted per row:", y_pred.sum(axis=1).mean())
    print(attr, "- avg true labels per row:", y_test.sum(axis=1).mean())

embellishment - avg labels predicted per row: 0.0
embellishment - avg true labels per row: 1.0909090909090908
color - avg labels predicted per row: 0.0
color - avg true labels per row: 1.0


In [ ]:
import numpy as np
def predict_multilabel(model, X_input, threshold=0.3):
    probas = model.predict_proba(X_input)  # shape: (n_samples, n_labels)
    preds = (probas >= threshold).astype(int)
    # fallback: if a row has zero labels predicted, force the single highest-probability label
    for i in range(preds.shape[0]):
        if preds[i].sum() == 0:
            top_label_idx = np.argmax(probas[i])
            preds[i, top_label_idx] = 1
    return preds

In [ ]:
multi_label_results = []
for attr in multi_label_attrs:
    mlb = multi_label_binarizers[attr]
    y = mlb.transform(df[attr])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = multi_label_models[attr]
    y_pred = predict_multilabel(model, X_test, threshold=0.3)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
    multi_label_results.append({"attribute": attr, "accuracy": acc, "f1_score": f1})
multi_label_df = pd.DataFrame(multi_label_results)
print(multi_label_df)

       attribute  accuracy  f1_score
0  embellishment  0.272727  0.347826
1          color  0.272727  0.272727


In [ ]:
all_results = pd.concat([single_label_df, multi_label_df], ignore_index=True)
print(all_results)

overall_f1 = all_results["f1_score"].mean()
print("Overall average F1 score across all attributes:", overall_f1)
all_results.to_csv("evaluation_metrics.csv", index=False)

       attribute  accuracy  f1_score
0       category  0.727273  0.628099
1     silhouette  0.363636  0.223776
2         fabric  0.454545  0.353535
3       neckline  0.727273  0.721212
4         sleeve  0.454545  0.450000
5         length  0.363636  0.216783
6  embellishment  0.272727  0.347826
7          color  0.272727  0.272727
Overall average F1 score across all attributes: 0.4017449310680536
